In [1]:
import sys
from pathlib import Path 

project_root = Path.cwd().parent 
sys.path.insert(0, str(project_root))

from src.evaluate_models import eval_reg

import pandas as pd 
import numpy as np 

from sklearn.model_selection import train_test_split   
from sklearn.tree import DecisionTreeRegressor

print("Session 27 imports greatly successful")

Session 27 imports greatly successful


In [2]:
df = pd.read_csv(
    "../data/student-mat.csv",
    sep=";"
)

df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

X_full = df_encoded.drop(
    columns=["G3"]
).copy()

y = df_encoded["G3"].copy()

print("Dataset loaded.")
print("X_full shape:", X_full.shape)

Dataset loaded.
X_full shape: (395, 41)


In [3]:
Xtr_f, Xte_f, ytr, yte = train_test_split(
    X_full,
    y,
    test_size=0.20,
    random_state=42
)

print("Train/test split complete.")
print("Training rows:", len(Xtr_f))
print("Test rows:", len(Xte_f))

Train/test split complete.
Training rows: 316
Test rows: 79


In [4]:
tree = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

tree.fit(
    Xtr_f,
    ytr
)

print("Decision Tree trained successfully.")

Decision Tree trained successfully.


In [5]:
tree_predictions = tree.predict(
    Xte_f
)

print(
    "Predictions generated:",
    len(tree_predictions)
)

Predictions generated: 79


In [6]:
tree_test_results = eval_reg(
    yte,
    tree_predictions
)

print("Decision Tree test results:")
print(tree_test_results)

Decision Tree test results:
{'MAE': 1.357218203737191, 'RMSE': 2.467248497769575, 'R2': 0.7031308891822728}


In [7]:
print("Configured max_depth:", tree.max_depth)
print("Actual fitted depth:", tree.get_depth())
print("Number of leaves:", tree.get_n_leaves())

Configured max_depth: 5
Actual fitted depth: 5
Number of leaves: 31


In [8]:
decision_tree_row = pd.DataFrame([
    {
        "Model": "Decision Tree",
        "MAE": tree_test_results["MAE"],
        "RMSE": tree_test_results["RMSE"],
        "R2": tree_test_results["R2"]
    }
])

decision_tree_row.round(4)

,Model,MAE,RMSE,R2
0,Decision Tree,1.3572,2.4672,0.7031


In [9]:
model_comparison_df = pd.DataFrame([
    {
        "Model": "KNN",
        "MAE": 2.5848101265822785,
        "RMSE": 3.3926950565642415,
        "R2": 0.4386562685587473
    },
    {
        "Model": "SVR",
        "MAE": 1.8366678536589514,
        "RMSE": 2.7260297218073055,
        "R2": 0.6375898115704413
    },
    {
        "Model": "Decision Tree",
        "MAE": tree_test_results["MAE"],
        "RMSE": tree_test_results["RMSE"],
        "R2": tree_test_results["R2"]
    }
])

model_comparison_df.round(4)

,Model,MAE,RMSE,R2
0,KNN,2.5848,3.3927,0.4387
1,SVR,1.8367,2.7260,0.6376
2,Decision Tree,1.3572,2.4672,0.7031


In [10]:
model_comparison_df = model_comparison_df.sort_values(
    by="RMSE",
    ascending=True
).reset_index(drop=True)

model_comparison_df["RMSE_Rank"] = (
    model_comparison_df["RMSE"]
    .rank(method="min", ascending=True)
    .astype(int)
)

model_comparison_df

,Model,MAE,RMSE,R2,RMSE_Rank
0,Decision Tree,1.357218,2.467248,0.703131,1
1,SVR,1.836668,2.726030,0.637590,2
2,KNN,2.584810,3.392695,0.438656,3


In [11]:
decision_tree_table_row = model_comparison_df[
    model_comparison_df["Model"] == "Decision Tree"
]

print(decision_tree_table_row.round(4))

           Model     MAE    RMSE      R2  RMSE_Rank
0  Decision Tree  1.3572  2.4672  0.7031          1
